# NB32 — Relative Functional Enrichment (RFE): Driver vs. Passenger Test

**Motivation:** Arc 1 shows that cofactor gene density (cobalamin pathway) predicts niche specialisation
(β = −0.033, p < 10⁻⁸), but translation (housekeeping) gene density predicts it equally well
(β = −0.030, ρ(cofactor, translation) = 0.364). In a joint model, cofactor loses significance.
This could mean cofactor genes are mere *passengers* on compact genomes — not drivers of
specialisation in their own right.

**Test:** Replace absolute density with the **Relative Functional Enrichment (RFE)** ratio:

$$\text{log\_RFE} = \log(\text{cobalamin\_per\_Mb} + \varepsilon) - \log(\text{translation\_per\_Mb})$$

**Null (passenger):** Genera with more cobalamin genes also have proportionally more translation
genes (both respond to genome compactness). The ratio is constant; β(RFE) ≈ 0.

**Alternative (driver):** Genera that are *disproportionately* enriched in cobalamin relative to
translation are the narrowest specialists. β(RFE) < 0, significant after genome-size control.

**Models run:**
- M0 (reference): `B_std ~ ko_per_mb_z` — primary Arc 1 result
- M1: `B_std ~ cobalamin_z` — cobalamin standalone
- M2: `B_std ~ translation_z` — translation standalone
- M3: `B_std ~ cobalamin_z + translation_z` — joint model (expected collinearity failure)
- M4: `B_std ~ rfe_z` — RFE standalone (main driver test)
- M5: `B_std ~ rfe_z + genome_mb_z` — RFE + genome size (passenger control)


In [1]:
import sys
from pathlib import Path

_project_root = Path().resolve().parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H
apply_style()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

from scripts.pgls_utils import run_pgls, run_multi_pgls, pgls_results_table

DATA   = _project_root / 'data'
FIGS   = _project_root / 'figures'
TREE   = DATA / 'gtdb_bac_genus_pruned.tree'

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK — project root:', _project_root)

Imports OK — project root: /home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology


In [2]:
# ── Load and join ───────────────────────────────────────────────────────────
df_pgls  = pd.read_csv(DATA / '01_pgls_input_bacteria.csv')
df_cof   = pd.read_csv(DATA / 'expanded_kegg_metal_cofactor_densities.csv')
df_trans = pd.read_csv(DATA / 'landscape_translation_density.csv')

# Rename translation density column for clarity
df_trans = df_trans.rename(columns={'ko_per_mb': 'translation_per_mb'})

# Inner join: restrict to PGLS genera that have both cobalamin and translation data
df = (
    df_pgls
    .merge(df_cof[['genus_lower', 'cobalamin_per_mb', 'cobalamin_z']], on='genus_lower', how='inner')
    .merge(df_trans[['genus_lower', 'translation_per_mb']], on='genus_lower', how='inner')
)

print(f'Joined table: {df.shape[0]} genera x {df.shape[1]} columns')
print(f'Genera with cobalamin_per_mb == 0: {(df.cobalamin_per_mb == 0).sum()}')
print(f'cobalamin_per_mb  — median: {df.cobalamin_per_mb.median():.3f}  '
      f'min nonzero: {df.cobalamin_per_mb[df.cobalamin_per_mb > 0].min():.4f}')
print(f'translation_per_mb — median: {df.translation_per_mb.median():.3f}')

Joined table: 1073 genera x 11 columns
Genera with cobalamin_per_mb == 0: 226
cobalamin_per_mb  — median: 1.483  min nonzero: 0.1280
translation_per_mb — median: 13.595


In [3]:
# ── Compute log-RFE ─────────────────────────────────────────────────────────
# Pseudo-count for zero cobalamin: 0.01 KO/Mb (well below min nonzero ~0.128)
EPS = 0.01

df['log_cobalamin']    = np.log(df['cobalamin_per_mb'] + EPS)
df['log_translation']  = np.log(df['translation_per_mb'])          # never zero
df['log_rfe']          = df['log_cobalamin'] - df['log_translation']

# Standardise all predictors (z-score)
def zscore(s):
    return (s - s.mean()) / s.std()

df['cobalamin_z2']    = zscore(df['cobalamin_per_mb'])  # re-derived from this join
df['translation_z']   = zscore(df['translation_per_mb'])
df['rfe_z']           = zscore(df['log_rfe'])

# genome_mb_z is already in the PGLS input; verify it's present
assert 'genome_mb_z' in df.columns

print('RFE descriptives:')
print(df[['log_rfe', 'rfe_z']].describe().round(3))
print(f'\nCorr(cobalamin_per_mb, translation_per_mb): '
      f'{df.cobalamin_per_mb.corr(df.translation_per_mb):.3f}')
print(f'Corr(cobalamin_z2, translation_z):          '
      f'{df.cobalamin_z2.corr(df.translation_z):.3f}')

RFE descriptives:
        log_rfe     rfe_z
count 1073.0000 1073.0000
mean    -3.2200   -0.0000
std      2.3310    1.0000
min     -8.9080   -2.4400
25%     -3.8990   -0.2910
50%     -1.9740    0.5340
75%     -1.5620    0.7110
max      0.0330    1.3950

Corr(cobalamin_per_mb, translation_per_mb): 0.002
Corr(cobalamin_z2, translation_z):          0.002


In [4]:
# ── Run all PGLS models ──────────────────────────────────────────────────────
RESPONSE = 'mean_levins_B_std'

model_specs = [
    ('M0_primary',       ['predictor_z']),
    ('M1_cobalamin',     ['cobalamin_z2']),
    ('M2_translation',   ['translation_z']),
    ('M3_joint',         ['cobalamin_z2', 'translation_z']),
    ('M4_rfe',           ['rfe_z']),
    ('M5_rfe_genomesize',['rfe_z', 'genome_mb_z']),
]

results = []
for label, preds in model_specs:
    print(f'Running {label} ...')
    try:
        res = run_pgls(df, TREE, response=RESPONSE, predictors=preds, label=label)
        results.append(res)
        # Print quick summary
        for pred in preds:
            b  = res['betas'][pred]
            se = res['SEs'][pred]
            p  = res['p_values'][pred]
            print(f'  {pred:25s}  β={b:+.4f}  SE={se:.4f}  p={p:.4e}')
        print(f'  λ={res["lambda_est"]:.3f}  ΔAIC={res["delta_aic_vs_null"]:.1f}  n={res["n"]}')
    except Exception as exc:
        print(f'  ERROR: {exc}')
        results.append({'label': label, 'converged': False, 'error': str(exc)})

print('\nAll models done.')

Running M0_primary ...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  predictor_z                β=-0.0223  SE=0.0041  p=4.6647e-08
  λ=0.789  ΔAIC=-27.9  n=1073
Running M1_cobalamin ...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  cobalamin_z2               β=-0.0193  SE=0.0046  p=2.8985e-05
  λ=0.807  ΔAIC=-15.5  n=1073
Running M2_translation ...


...

  cobalamin_z2               β=-0.0180  SE=0.0045  p=7.1910e-05
  translation_z              β=-0.0290  SE=0.0047  p=1.0634e-09
  λ=0.787  ΔAIC=-51.0  n=1073
Running M4_rfe ...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  rfe_z                      β=-0.0046  SE=0.0048  p=3.3360e-01
  λ=0.811  ΔAIC=1.1  n=1073
Running M5_rfe_genomesize ...


/home/hmacgregor/.local/lib/python3.13/site-packages/dendropy/datamodel/treemodel/_tree.py:1510: UserWarning: Calculating MRCA on an unrooted tree implicitly implicitly treats seed node as root. Set tree.is_rooted = True to silence this warning.
  warnings.warn(


  rfe_z                      β=-0.0133  SE=0.0048  p=5.5923e-03
  genome_mb_z                β=+0.0404  SE=0.0049  p=8.8818e-16
  λ=0.784  ΔAIC=-61.6  n=1073

All models done.


In [5]:
# ── Results table ─────────────────────────────────────────────────────────────
tbl = pgls_results_table(results)

# Mark significance
def sig_stars(p):
    if pd.isna(p):    return ''
    if p < 0.001:     return '***'
    if p < 0.01:      return '**'
    if p < 0.05:      return '*'
    return 'ns'

tbl['sig'] = tbl['p_value'].apply(sig_stars)

display_cols = ['label', 'predictor', 'n', 'lambda_est', 'beta', 'SE', 'p_value', 'sig', 'delta_aic_vs_null']
print(tbl[display_cols].to_string(index=False))

# Save
tbl.to_csv(DATA / '32_rfe_pgls_results.csv', index=False)
print(f'\nSaved: data/32_rfe_pgls_results.csv')

            label     predictor    n  lambda_est    beta     SE  p_value sig  delta_aic_vs_null
       M0_primary   predictor_z 1073      0.7894 -0.0223 0.0041   0.0000 ***           -27.9200
     M1_cobalamin  cobalamin_z2 1073      0.8075 -0.0193 0.0046   0.0000 ***           -15.5200
   M2_translation translation_z 1073      0.7900 -0.0299 0.0047   0.0000 ***           -37.0400
         M3_joint  cobalamin_z2 1073      0.7868 -0.0180 0.0045   0.0001 ***           -50.9700
         M3_joint translation_z 1073      0.7868 -0.0290 0.0047   0.0000 ***           -50.9700
           M4_rfe         rfe_z 1073      0.8115 -0.0046 0.0048   0.3336  ns             1.0600
M5_rfe_genomesize         rfe_z 1073      0.7839 -0.0133 0.0048   0.0056  **           -61.6200
M5_rfe_genomesize   genome_mb_z 1073      0.7839  0.0404 0.0049   0.0000 ***           -61.6200

Saved: data/32_rfe_pgls_results.csv


In [6]:
# ── Verbal verdict ────────────────────────────────────────────────────────────
m4 = next((r for r in results if r.get('label') == 'M4_rfe' and r.get('converged', True)), None)
m5 = next((r for r in results if r.get('label') == 'M5_rfe_genomesize' and r.get('converged', True)), None)

if m4 and 'p_values' in m4:
    p4 = m4['p_values']['rfe_z']
    b4 = m4['betas']['rfe_z']
    p5 = m5['p_values']['rfe_z'] if m5 and 'p_values' in m5 else float('nan')
    b5 = m5['betas']['rfe_z']    if m5 and 'betas' in m5    else float('nan')
    print('=== DRIVER vs PASSENGER VERDICT ===')
    print(f'M4 (RFE alone):          β(rfe_z)={b4:+.4f}, p={p4:.4e}')
    print(f'M5 (RFE + genome size):  β(rfe_z)={b5:+.4f}, p={p5:.4e}')
    if p5 < 0.05 and b5 < 0:
        print('VERDICT: DRIVER — RFE significant after genome-size control.')
        print('  Genera enriched in cobalamin relative to translation are narrower'
              ' specialists, independent of genome compactness.')
    elif p4 < 0.05 and p5 >= 0.05:
        print('VERDICT: PASSENGER (marginal) — RFE significant alone but not after'
              ' genome-size control.')
        print('  The cofactor signal may reflect genome compactness rather than'
              ' cobalamin-specific specialisation.')
    elif p4 >= 0.05:
        print('VERDICT: PASSENGER — RFE is not significant. The cofactor:translation'
              ' ratio does not predict niche breadth beyond genome size.')
    else:
        print(f'Ambiguous — interpret raw numbers above.')

=== DRIVER vs PASSENGER VERDICT ===
M4 (RFE alone):          β(rfe_z)=-0.0046, p=3.3360e-01
M5 (RFE + genome size):  β(rfe_z)=-0.0133, p=5.5923e-03
VERDICT: DRIVER — RFE significant after genome-size control.
  Genera enriched in cobalamin relative to translation are narrower specialists, independent of genome compactness.


In [7]:
# ── Figure 1: Model coefficient comparison ────────────────────────────────────
tbl_plot = tbl.copy()
tbl_plot = tbl_plot[tbl_plot['predictor'].isin(['predictor_z','cobalamin_z2','translation_z','rfe_z'])]

# Order by model (M0 → M5)
order_map = {
    ('M0_primary', 'predictor_z'):       ('M0 — 140-KO metal\n(Arc 1 reference)', 0),
    ('M1_cobalamin', 'cobalamin_z2'):    ('M1 — Cobalamin only',                  1),
    ('M2_translation', 'translation_z'): ('M2 — Translation only',                2),
    ('M3_joint', 'cobalamin_z2'):        ('M3 — Joint: cobalamin',                3),
    ('M3_joint', 'translation_z'):       ('M3 — Joint: translation',              4),
    ('M4_rfe', 'rfe_z'):                 ('M4 — RFE alone',                       5),
    ('M5_rfe_genomesize', 'rfe_z'):      ('M5 — RFE + genome size',               6),
}

rows = []
for _, row in tbl_plot.iterrows():
    key = (row['label'], row['predictor'])
    if key in order_map:
        lbl, idx = order_map[key]
        rows.append({'label': lbl, 'idx': idx, 'beta': row['beta'],
                     'SE': row['SE'], 'p': row['p_value']})

rows = sorted(rows, key=lambda r: r['idx'])
labels = [r['label'] for r in rows]
betas  = [r['beta']  for r in rows]
ses    = [r['SE']    for r in rows]
pvals  = [r['p']     for r in rows]
colors = [PALETTE[0] if r['p'] < 0.05 else PALETTE[5] for r in rows]

fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))
y_pos = np.arange(len(labels))[::-1]

ax.barh(y_pos, betas, xerr=ses, color=colors, edgecolor='k', linewidth=0.5,
        error_kw=dict(elinewidth=0.8, ecolor='k', capsize=3))
ax.axvline(0, color='gray', lw=0.8, ls='--')
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel('PGLS β (± 1 SE)', fontsize=9)
ax.set_ylabel('')
ax.set_title('Cobalamin vs. translation density: model comparison', fontsize=10)

# Significance annotation
for yi, row in zip(y_pos, rows):
    stars = sig_stars(row['p'])
    x_ann = row['beta'] + row['SE'] + 0.0005
    ax.annotate(stars, (x_ann, yi), va='center', ha='left', fontsize=8, color='#808080')

fig.suptitle('Driver vs. Passenger: Relative Functional Enrichment', y=1.02, fontsize=11, fontweight='bold')
save(fig, FIGS / 'fig_nb32_rfe_model_comparison')
print('Saved fig_nb32_rfe_model_comparison.pdf')

Saved fig_nb32_rfe_model_comparison.pdf


In [8]:
# ── Figure 2: RFE vs niche breadth scatter (coloured by phylum) ──────────────
TOP_PHYLA = (
    df['phylum']
    .value_counts()
    .head(6)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(FIGW['1.5col'], ROW_H))

for ci, phy in enumerate(TOP_PHYLA):
    sub = df[df['phylum'] == phy]
    ax.scatter(sub['log_rfe'], sub['mean_levins_B_std'],
               color=PALETTE[ci % len(PALETTE)], s=8, alpha=0.6,
               label=phy, zorder=2)

# Other phyla in gray
other = df[~df['phylum'].isin(TOP_PHYLA)]
ax.scatter(other['log_rfe'], other['mean_levins_B_std'],
           color='#cccccc', s=5, alpha=0.4, label='Other', zorder=1)

# PGLS regression line (M4 — RFE alone)
if m4 and 'betas' in m4:
    intercept = next(
        (r for r in results if r.get('label') == 'M4_rfe'),
        {}
    )
    # Plot OLS trend line as visual guide (PGLS line would require VCV; OLS is approximate)
    slope_ols, inter_ols, *_ = stats.linregress(df['rfe_z'], df['mean_levins_B_std'])
    x_line = np.linspace(df['log_rfe'].min(), df['log_rfe'].max(), 100)
    x_line_z = (x_line - df['log_rfe'].mean()) / df['log_rfe'].std()
    ax.plot(x_line, inter_ols + slope_ols * x_line_z,
            color='k', lw=1.2, ls='-', zorder=5, label='OLS trend')

ax.set_xlabel('log(Cobalamin/Mb) − log(Translation/Mb)  [log-RFE]', fontsize=9)
ax.set_ylabel('Levins\' B (standardised)', fontsize=9)
ax.set_title('Relative cobalamin enrichment vs. niche breadth', fontsize=10)
ax.legend(fontsize=7, frameon=False, ncol=2, loc='upper right')

if m4 and 'p_values' in m4:
    p_txt = m4['p_values']['rfe_z']
    b_txt = m4['betas']['rfe_z']
    ax.annotate(f'M4 β={b_txt:+.3f}\np={p_txt:.2e}',
                xy=(0.05, 0.05), xycoords='axes fraction',
                fontsize=8, color='#808080', va='bottom')

fig.suptitle('Driver vs. Passenger: cobalamin:translation enrichment', y=1.02,
             fontsize=11, fontweight='bold')
save(fig, FIGS / 'fig_nb32_rfe_scatter')
print('Saved fig_nb32_rfe_scatter.pdf')

Saved fig_nb32_rfe_scatter.pdf


In [9]:
# ── Figure 3: Cobalamin vs Translation density coloured by RFE quartile ──────
df['rfe_quartile'] = pd.qcut(df['log_rfe'], q=4, labels=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])

fig, ax = plt.subplots(figsize=(FIGW['1col'], ROW_H))

quartile_colors = [PALETTE[5], PALETTE[4], PALETTE[1], PALETTE[0]]
for qi, (qlab, qcolor) in enumerate(zip(['Q1 (low)','Q2','Q3','Q4 (high)'], quartile_colors)):
    sub = df[df['rfe_quartile'] == qlab]
    ax.scatter(sub['translation_per_mb'], sub['cobalamin_per_mb'],
               color=qcolor, s=7, alpha=0.6, label=f'RFE {qlab}', zorder=2)

ax.set_xlabel('Translation gene density (KO/Mb)', fontsize=9)
ax.set_ylabel('Cobalamin gene density (KO/Mb)', fontsize=9)
ax.set_title('Cobalamin vs. Translation density\ncoloured by RFE quartile', fontsize=10)
ax.legend(fontsize=7, frameon=False)
ax.set_xscale('log')
ax.set_yscale('log')

save(fig, FIGS / 'fig_nb32_cobalamin_vs_translation')
print('Saved fig_nb32_cobalamin_vs_translation.pdf')

Saved fig_nb32_cobalamin_vs_translation.pdf


In [10]:
# ── Final summary ─────────────────────────────────────────────────────────────
print('=== NB32 SUMMARY ===')
print(f'N genera in analysis: {len(df)}')
print(f'N genera with cobalamin=0 (pseudo-count applied): {(df.cobalamin_per_mb == 0).sum()}')
print(f'Cobalamin:translation correlation: {df.cobalamin_per_mb.corr(df.translation_per_mb):.3f}')
print()

for r in results:
    if not r.get('converged', True) or 'p_values' not in r:
        print(f'{r["label"]:30s}  ERROR: {r.get("error","?")}')
        continue
    for pred in r['predictors']:
        b = r['betas'][pred];  p = r['p_values'][pred]
        print(f'{r["label"]:30s}  {pred:25s}  β={b:+.4f}  p={p:.3e}  {sig_stars(p)}')

print()
print('Outputs:')
print('  data/32_rfe_pgls_results.csv')
print('  figures/fig_nb32_rfe_model_comparison.pdf')
print('  figures/fig_nb32_rfe_scatter.pdf')
print('  figures/fig_nb32_cobalamin_vs_translation.pdf')

=== NB32 SUMMARY ===
N genera in analysis: 1073
N genera with cobalamin=0 (pseudo-count applied): 226
Cobalamin:translation correlation: 0.002

M0_primary                      predictor_z                β=-0.0223  p=4.665e-08  ***
M1_cobalamin                    cobalamin_z2               β=-0.0193  p=2.898e-05  ***
M2_translation                  translation_z              β=-0.0299  p=4.357e-10  ***
M3_joint                        cobalamin_z2               β=-0.0180  p=7.191e-05  ***
M3_joint                        translation_z              β=-0.0290  p=1.063e-09  ***
M4_rfe                          rfe_z                      β=-0.0046  p=3.336e-01  ns
M5_rfe_genomesize               rfe_z                      β=-0.0133  p=5.592e-03  **
M5_rfe_genomesize               genome_mb_z                β=+0.0404  p=8.882e-16  ***

Outputs:
  data/32_rfe_pgls_results.csv
  figures/fig_nb32_rfe_model_comparison.pdf
  figures/fig_nb32_rfe_scatter.pdf
  figures/fig_nb32_cobalamin_vs_translatio